### 1. Load and preprocess data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("2026-04-26-atp_matches_best_of_3_ELO.csv")  # or read_rds equivalent via pyreadr

# type conversions (like mutate + as.factor in R)
categorical_cols = ["surface", "player1_hand", "player2_hand"]

for col in categorical_cols:
    df[col] = df[col].astype("category")

df["match_outcome"] = df["match_outcome"].astype("category")


# drop NA outcome
df = df.dropna(subset=["match_outcome"])

In [2]:
df['match_outcome'].value_counts() / df.shape[0]

match_outcome
WW     0.441163
LL     0.195667
LWW    0.114751
WLW    0.099227
WLL    0.080072
LWL    0.069121
Name: count, dtype: float64

### 2. Train/test split

In [2]:
from sklearn.model_selection import train_test_split

X = df[
    ["rank_diff", "elo_diff", "age_diff", "ht_diff",
     "surface", "player1_hand", "player2_hand"]
]
y = df["match_outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

### 3. Preprocessing pipeline

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = ["rank_diff", "elo_diff", "age_diff", "ht_diff"]
categorical_features = ["surface", "player1_hand", "player2_hand"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first"), categorical_features)
    ]
)

### 4. Multinomial Logistic Regression with class weights

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = LogisticRegression(
    solver="lbfgs",
    class_weight="balanced",   # <-- THIS is the key equivalent
    max_iter=1000
)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

### 5. Cross-validation

In [5]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

cv_acc = cross_val_score(clf, X_train, y_train, cv=cv, scoring="accuracy")



### 6. Fit the model

In [6]:
clf.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['rank_diff', 'elo_diff',
                                                   'age_diff', 'ht_diff']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['surface', 'player1_hand',
                                                   'player2_hand'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

### 7. Predictions on test set

In [7]:
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)

### 8. Metrics (accuracy, confusion matrix)

In [8]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob, multi_class="ovr", average="weighted")
acc_auc_summary = pd.DataFrame({
    "metric" : ["accuracy", "roc auc score"],
    "estimate": [acc, auc]
})

acc_auc_summary

,metric,estimate
0,accuracy,0.332548
1,roc auc score,0.610559


In [9]:
cm = confusion_matrix(y_test, y_pred)

# 1. Long-format truth vs prediction table
df_cm = pd.DataFrame({"truth": y_test, "prediction": y_pred})

# 2. Confusion matrix as a clean table (counts)
cm_table = pd.crosstab(
    df_cm["truth"],
    df_cm["prediction"],
    rownames=["Truth"],
    colnames=["Prediction"]
)

print("Confusion matrix:")
cm_table

Confusion matrix:


Prediction,LL,LWL,LWW,WLL,WLW,WW
Truth,,,,,,
LL,1020,238,66,121,168,381
LWL,330,80,30,36,62,166
LWW,391,104,36,80,114,445
WLL,359,99,27,51,63,217
WLW,322,104,43,64,104,374
WW,1176,404,162,232,424,2098


### 9. Log loss

In [10]:
from sklearn.metrics import log_loss

ll = log_loss(y_test, y_prob)
ll

1.7366960061735213

### 10. Per-class accuracy

In [11]:
cm = confusion_matrix(y_test, y_pred)
per_class_acc = cm.diagonal() / cm.sum(axis=1)
results = pd.DataFrame({
    "class": clf.classes_,
    "accuracy": per_class_acc
})
results



,class,accuracy
0,LL,0.511535
1,LWL,0.113636
2,LWW,0.030769
3,WLL,0.062500
4,WLW,0.102868
5,WW,0.466637


### 11. Calibration

In [12]:
# Step 1: Build calibration dataframe (long format equivalent of pivot_longer)
classes = clf.classes_
calibration_df = pd.DataFrame(y_prob, columns=classes)
calibration_df["actual"] = y_test.values

# equivalent of pivot_longer
calibration_long = calibration_df.melt(
    id_vars="actual",
    var_name="class",
    value_name="prob"
)

# Step 2: Ensure type consistency (like gsub + as.character)
calibration_long["actual"] = calibration_long["actual"].astype(str)
calibration_long["class"] = calibration_long["class"].astype(str)

# Step 3: Calculate calibration 
multi_calibration = (
    calibration_long
    .groupby("class")
    .apply(lambda df: pd.Series({
        "observed": (df["actual"] == df["class"]).sum(),
        "total_pred_prob": df["prob"].sum()
    }))
    .reset_index()
)

multi_calibration["calibration"] = (
    multi_calibration["total_pred_prob"] /
    multi_calibration["observed"].replace(0, np.nan)
)

multi_calibration


/var/folders/62/8_hqgqn96wq06y7v4pgj6_kh0000gn/T/ipykernel_44389/2894591724.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  calibration_long


,class,observed,total_pred_prob,calibration
0,LL,1994.0,1638.975336,0.821954
1,LWL,704.0,1655.194978,2.351129
2,LWW,1170.0,1722.634232,1.472337
3,WLL,816.0,1659.644295,2.033878
4,WLW,1011.0,1730.087251,1.711263
5,WW,4496.0,1784.463906,0.396900


### SUMMARY OF PERFORMANCE

In [13]:
perf_summary = pd.DataFrame({
    "metric": ["accuracy", "calibration", "log-loss"],
    "estimate": [acc, multi_calibration['calibration'].mean(), ll]
})

perf_summary

,metric,estimate
0,accuracy,0.332548
1,calibration,1.464577
2,log-loss,1.736696
